In [14]:
from pathlib import Path
import os
print("cwd:", Path.cwd())
# set to repo root explicitly (Windows path)
os.chdir(r"f:\End-to-end-machine-learning-project-with-mlflow")
print("new cwd:", Path.cwd())

cwd: f:\
new cwd: f:\End-to-end-machine-learning-project-with-mlflow


In [15]:
%pwd

'f:\\End-to-end-machine-learning-project-with-mlflow'

In [16]:
os.chdir("../")

In [17]:
%pwd

'f:\\'

In [18]:
from pathlib import Path
import os

# compute repo root reliably — __file__ isn't available in notebooks, use a fallback
if "__file__" in globals():
    ROOT_DIR = Path(__file__).resolve().parents[2]
else:
    # prefer the current working dir (you already set it at the top of the notebook)
    cwd = Path.cwd()
    def find_repo_root(start: Path):
        for p in [start] + list(start.parents):
            # look for common repo markers
            if (p / "config").exists() or (p / ".git").exists() or (p / "setup.py").exists():
                return p
        return start
    ROOT_DIR = find_repo_root(cwd)

CONFIG_DIR = ROOT_DIR / "config"
CONFIG_FILE_PATH = CONFIG_DIR / "config.yaml"
PARAMS_FILE_PATH = CONFIG_DIR / "params.yaml"
SCHEMA_FILE_PATH = CONFIG_DIR / "schema.yaml"

In [19]:
from pathlib import Path
print(ROOT_DIR, (CONFIG_FILE_PATH).exists(), CONFIG_FILE_PATH)

f:\ False f:\config\config.yaml


In [20]:
CONFIG_FILE_PATH = CONFIG_DIR / "config.yaml"  # CONFIG_DIR not defined yet!

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import urllib.request as request
import zipfile

from mlProject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from mlProject.utils.common import read_yaml, create_directories, get_size
from mlProject import logger

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        artifacts_root = self.config.get("artifacts_root", "artifacts")
        create_directories([Path(artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        cfg = self.config.get("data_ingestion", {})
        root_dir = Path(cfg.get("root_dir", "artifacts/data_ingestion"))
        source_url = cfg.get("source_url", "")
        local_data_file = Path(cfg.get("local_data_file", root_dir / "data.zip"))
        unzip_dir = Path(cfg.get("unzip_dir", root_dir / "extracted"))

        create_directories([root_dir, unzip_dir])

        return DataIngestionConfig(
            root_dir=root_dir,
            source_url=source_url,
            local_data_file=local_data_file,
            unzip_dir=unzip_dir,
        )

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> Path:
        target = Path(self.config.local_data_file)
        target.parent.mkdir(parents=True, exist_ok=True)

        if not target.exists():
            logger.info(f"Downloading {self.config.source_url} -> {target}")
            try:
                filename, headers = request.urlretrieve(
                    url=self.config.source_url,
                    filename=str(target)
                )
                logger.info(f"Downloaded to {filename}; headers: {headers}")
            except Exception:
                logger.exception("Failed to download data")
                raise
        else:
            logger.info(f"File already exists: {target} size={get_size(target)}")

        return target

    def extract_zip_file(self, zip_file_path: Path | str = None, extract_to: Path | str = None) -> Path:
        zip_file_path = Path(zip_file_path or self.config.local_data_file)
        extract_to = Path(extract_to or self.config.unzip_dir)
        extract_to.mkdir(parents=True, exist_ok=True)

        if not zip_file_path.exists():
            raise FileNotFoundError(f"Zip file not found: {zip_file_path}")

        logger.info(f"Extracting {zip_file_path} -> {extract_to}")
        try:
            with zipfile.ZipFile(str(zip_file_path), 'r') as zip_ref:
                zip_ref.extractall(str(extract_to))
        except Exception:
            logger.exception("Failed to extract zip")
            raise

        return extract_to

    def initiate_data_ingestion(self) -> Path:
        zip_path = self.download_file()
        extracted_dir = self.extract_zip_file(zip_path, self.config.unzip_dir)
        return extracted_dir

# Usage
try:
    cfg_mgr = ConfigurationManager()
    di_cfg = cfg_mgr.get_data_ingestion_config()
    di = DataIngestion(di_cfg)
    extracted = di.initiate_data_ingestion()
    logger.info(f"Data ingestion completed, extracted to: {extracted}")
except Exception as e:
    logger.exception("Data ingestion failed")
    raise

[2025-11-30 17:26:28,828: ERROR: 3271222918: Data ingestion failed]
Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_13656\3271222918.py", line 97, in <module>
    cfg_mgr = ConfigurationManager()
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_13656\3271222918.py", line 25, in __init__
    self.config = read_yaml(config_filepath)
  File "c:\Users\USER\anaconda3\envs\mlflow\lib\site-packages\ensure\main.py", line 872, in __call__
    return_val = self.f(*args, **kwargs)
  File "F:\End-to-end-machine-learning-project-with-mlflow\src\mlProject\utils\common.py", line 36, in read_yaml
    raise e
  File "F:\End-to-end-machine-learning-project-with-mlflow\src\mlProject\utils\common.py", line 29, in read_yaml
    with open(path_to_yaml) as yaml_file:
FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'


FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'

In [21]:
from dataclasses import dataclass
from pathlib import Path
import os


CONFIG_FILE_PATH = CONFIG_DIR / "config.yaml"
PARAMS_FILE_PATH = CONFIG_DIR / "params.yaml"
SCHEMA_FILE_PATH = CONFIG_DIR / "schema.yaml"

import urllib.request as request
import zipfile

from mlProject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from mlProject.utils.common import read_yaml, create_directories, get_size
from mlProject import logger

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

class ConfigurationManager:
    def __init__(
        self,
        config_filepath: str = CONFIG_FILE_PATH,
        params_filepath: str = PARAMS_FILE_PATH,
        schema_filepath: str = SCHEMA_FILE_PATH,
    ):
        # read_yaml should return a mapping (dict-like). use .get to be safe.
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        artifacts_root = self.config.get("artifacts_root", "artifacts")
        create_directories([Path(artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        cfg = self.config.get("data_ingestion", {})
        root_dir = Path(cfg.get("root_dir", "artifacts/data_ingestion"))
        source_url = cfg.get("source_url", "")
        local_data_file = Path(cfg.get("local_data_file", root_dir / "data.zip"))
        unzip_dir = Path(cfg.get("unzip_dir", root_dir / "extracted"))

        create_directories([root_dir, unzip_dir])

        return DataIngestionConfig(
            root_dir=root_dir,
            source_url=source_url,
            local_data_file=local_data_file,
            unzip_dir=unzip_dir,
        )

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> Path:
        target = Path(self.config.local_data_file)
        target.parent.mkdir(parents=True, exist_ok=True)

        if not target.exists():
            logger.info(f"Downloading {self.config.source_url} -> {target}")
            try:
                filename, headers = request.urlretrieve(
                    url=self.config.source_url,
                    filename=str(target)
                )
                logger.info(f"Downloaded to {filename}; headers: {headers}")
            except Exception:
                logger.exception("Failed to download data")
                raise
        else:
            logger.info(f"File already exists: {target} size={get_size(target)}")

        return target

    def extract_zip_file(self, zip_file_path: Path | str = None, extract_to: Path | str = None) -> Path:
        zip_file_path = Path(zip_file_path or self.config.local_data_file)
        extract_to = Path(extract_to or self.config.unzip_dir)
        extract_to.mkdir(parents=True, exist_ok=True)

        if not zip_file_path.exists():
            raise FileNotFoundError(f"Zip file not found: {zip_file_path}")

        logger.info(f"Extracting {zip_file_path} -> {extract_to}")
        try:
            with zipfile.ZipFile(str(zip_file_path), 'r') as zip_ref:
                zip_ref.extractall(str(extract_to))
        except Exception:
            logger.exception("Failed to extract zip")
            raise

        return extract_to

    def initiate_data_ingestion(self) -> Path:
        zip_path = self.download_file()
        extracted_dir = self.extract_zip_file(zip_path, self.config.unzip_dir)
        return extracted_dir

# usage (run after the cell above has executed)
try:
    cfg_mgr = ConfigurationManager()
    di_cfg = cfg_mgr.get_data_ingestion_config()
    di = DataIngestion(di_cfg)
    extracted = di.initiate_data_ingestion()
    logger.info(f"Data ingestion completed, extracted to: {extracted}")
except Exception as e:
    logger.exception("Data ingestion failed")
    raise

[2025-11-30 17:28:54,973: ERROR: 2810702483: Data ingestion failed]
Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_13656\2810702483.py", line 104, in <module>
    cfg_mgr = ConfigurationManager()
  File "C:\Users\USER\AppData\Local\Temp\ipykernel_13656\2810702483.py", line 32, in __init__
    self.config = read_yaml(config_filepath)
  File "c:\Users\USER\anaconda3\envs\mlflow\lib\site-packages\ensure\main.py", line 872, in __call__
    return_val = self.f(*args, **kwargs)
  File "F:\End-to-end-machine-learning-project-with-mlflow\src\mlProject\utils\common.py", line 36, in read_yaml
    raise e
  File "F:\End-to-end-machine-learning-project-with-mlflow\src\mlProject\utils\common.py", line 29, in read_yaml
    with open(path_to_yaml) as yaml_file:
FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'


FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'